# Декоратор - это вызываемый объект, который принимает другую функцию в качестве аргумента (декодируемую функцию).

In [12]:
# @decorate
# def target():
#     print('hello')

# ====

# def target():
#     print('hello')

# target = decorate(target)

In [15]:
def deco(func):
    def inner():
        print('running inner()')
    
    return inner

@deco
def target():
    print('running terget()')

target()

print(target)

running inner()
<function deco.<locals>.inner at 0x72d425abcae0>


- декоратор - это функция или другой вызываемый объект
- декоратор может заменить декорируемую функцию другой
- декораторы выполняются сразу после загрузки модуля - смотреть файл registration.py и 
import_registration_file.py и попробовать вызвать их по отдельно чтобы посмотреть как будет себя вести декораторы. И тогда будет
понятно почему внутри декоратора нужно написать wrapper то есть обертку.

In [16]:
b=6

def f1(a):
    print(a)
    print(b)

f1(3)

3
6


In [17]:
b=6

def f2(a):
    print(a)
    print(b)
    b = 9

f2(3)

3


UnboundLocalError: cannot access local variable 'b' where it is not associated with a value

In [18]:
b = 6

def f3(a):
    global b
    print(a)
    print(b)
    b = 9

f3(3)
print(b)

3
6
9


### Замыкание

In [19]:
class Averager:

    def __init__(self):
        self.series = []

    def __call__(self, new_value, *args, **kwds):
        self.series.append(new_value)
        total = sum(self.series)
        current_result = total / len(self.series)
        print(current_result)

avg = Averager()

avg(10)
avg(11)
avg(12)

10.0
10.5
11.0


In [33]:
def make_averager():
    series = []

    def averager(new_value):
        series.append(new_value)
        total = sum(series)
        current_result = total / len(series)
        print(current_result)
    
    return averager


avg = make_averager()
print(avg, type(avg), sep='\n')
print()

avg(10)
avg(11)
avg(27)
print(avg.__code__.co_varnames)
print(avg.__code__.co_freevars, end='\n\n')

print(avg.__closure__)
print(avg.__closure__[0].cell_contents)


<function make_averager.<locals>.averager at 0x72d425b1e840>
<class 'function'>

10.0
10.5
16.0
('new_value', 'total', 'current_result')
('series',)

(<cell at 0x72d425c3fb20: list object at 0x72d425e17940>,)
[10, 11, 27]


In [35]:
# чтобы это понять нужно знать замыкание и свободную переменную
def make_averager():
    count = 0
    total = 0

    def averager(new_value):
        count += 1
        total += new_value
        current_result = total / count
        print(current_result)
    
    return averager

avg = make_averager()
avg(10)

UnboundLocalError: cannot access local variable 'count' where it is not associated with a value

In [36]:
def make_averager():
    count = 0
    total = 0

    def averager(new_value):
        nonlocal count, total
        count += 1
        total += new_value
        current_result = total / count
        print(current_result)
    
    return averager

avg = make_averager()
avg(10)
avg(11)
avg(12)

10.0
10.5
11.0


In [38]:
import time

def clock(func):
    def clocked(*args, **kwargs):
        t0 = time.perf_counter()
        result = func(*args)
        elapsed = time.perf_counter() - t0
        print(f'taken time -> {elapsed}, func -> {func}, resulte -> {result}')
        return result
    return clocked


@clock
def snooze(seconds):
    time.sleep(seconds)

@clock
def factorial(n):
    return 1 if n < 2 else n * factorial(n-1)

print('*' * 40)
snooze(.123)
print('*' * 40)
factorial(6)

****************************************
taken time -> 0.12308735301485285, func -> <function snooze at 0x72d425939580>, resulte -> None
****************************************
taken time -> 5.74975274503231e-07, func -> <function factorial at 0x72d42593b920>, resulte -> 1
taken time -> 2.5822024326771498e-05, func -> <function factorial at 0x72d42593b920>, resulte -> 2
taken time -> 4.189799074083567e-05, func -> <function factorial at 0x72d42593b920>, resulte -> 6
taken time -> 6.353500066325068e-05, func -> <function factorial at 0x72d42593b920>, resulte -> 24
taken time -> 7.722701411694288e-05, func -> <function factorial at 0x72d42593b920>, resulte -> 120
taken time -> 9.478401625528932e-05, func -> <function factorial at 0x72d42593b920>, resulte -> 720


720

## decorator with params

In [9]:
import time

DEFAULT_FMT = '[{elapsed:0.8f}s] {name}({args}) -> {result}'

def clock(fmt=DEFAULT_FMT):
    def decorate(func):
        def clocked(*_args):
            t0 = time.perf_counter()
            result = func(*_args)
            elapsed = time.perf_counter() - t0
            name = func.__name__
            args = ', '.join(repr(arg) for arg in _args)
            print(fmt.format(**locals()))
            return result
        return clocked
    return decorate

@clock()
def snooze(seconds):
    
    time.sleep(seconds)

for i in range(3):
    snooze(.123)

[0.12316095s] snooze(0.123) -> None
[0.12315769s] snooze(0.123) -> None
[0.12311651s] snooze(0.123) -> None


In [10]:
@clock('{name} -> {result}, how many seconds got {elapsed}')
def snooze(seconds):
    
    time.sleep(seconds)

for i in range(3):
    snooze(.123)

snooze -> None, how many seconds got 0.12310130300465971
snooze -> None, how many seconds got 0.12310481898020953
snooze -> None, how many seconds got 0.12309203698532656


### write ahead decorator using class (Class Decorator)

In [13]:
import time

DEFAULT_FMT = '[{elapsed:0.8f}s] {name}({args}) -> {result}'

class clock:
    
    def __init__(self, fmt=DEFAULT_FMT):
        self.fmt = fmt
    
    def __call__(self, func):
        def clocked(*_args):
            t0 = time.perf_counter()
            result = func(*_args)
            elapsed = time.perf_counter() - t0
            name = func.__name__
            args = ', '.join(repr(arg) for arg in _args)
            print(self.fmt.format(**locals()))
            return result
        return clocked



@clock()
def snooze_cls(seconds):
    time.sleep(seconds)

@clock('{result} {elapsed} {args} {name}')
def snooze_cls2(seconds):
    time.sleep(seconds)



snooze_cls(.523)
snooze_cls2(.523)

[0.52313364s] snooze_cls(0.523) -> None
None 0.5231287640053779 0.523 snooze_cls2
